# SEC S7 Comment Tables + Metadata Explorer

This notebook inspects the SEC crawler output under `../data/bulk_downloads/sec/sec_2023`. We load the `mapper.jsonl` metadata, SEC CSV exports (notice/proposed_rule/rule/public_submission), and the JSON tables generated by `scraper.py`. The goal is to see which S7 dockets were scraped successfully and to produce joined DataFrames that combine crawler rows with richer agency metadata.

In [1]:
from __future__ import annotations
import json
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import pandas as pd

pd.set_option('display.max_columns', 50)
DATA_ROOT = Path('../data/bulk_downloads/sec/sec_2023')
SCRAPED_DIR = DATA_ROOT / 'scraped_tables'
MAPPER_PATH = DATA_ROOT / 'mapper.jsonl'
print(f'Using data root: {DATA_ROOT.resolve()}')

Using data root: /Users/spangher/Projects/stanford-research/rfi-research/regulations-demo/data/bulk_downloads/sec/sec_2023


## Load mapper entries

`mapper.jsonl` ties Regulations.gov document IDs to the SEC's internal `sec_file_id` values. We explode each JSONL entry so that every `sec_file_id` (including the old-style S7 identifiers) gets its own row and keep URL metadata for later diagnostics.

In [2]:
def load_mapper_records(path: Path) -> pd.DataFrame:
    records: List[Dict[str, str]] = []
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            block = json.loads(line)
            ids = block.get('sec_file_id') or []
            info_list = block.get('url_info') or [{}] * len(ids)
            for idx, sec_id in enumerate(ids):
                sec_id = (sec_id or '').strip().upper()
                if not sec_id:
                    continue
                info_block = info_list[idx] if idx < len(info_list) else {}
                records.append(
                    {
                        'file_no': sec_id,
                        'mapper_type': block.get('type'),
                        'docket_id': block.get('docket'),
                        'posted_date': block.get('posted_date'),
                        'primary_url': (info_block or {}).get('primary_url'),
                        'fallback_url': (info_block or {}).get('fallback_url'),
                        'url_note': (info_block or {}).get('note'),
                    }
                )
    mapper_df = pd.DataFrame(records).drop_duplicates()
    mapper_df['is_s7'] = mapper_df['file_no'].str.startswith('S7-')
    return mapper_df

mapper_df = load_mapper_records(MAPPER_PATH)
s7_mapper = mapper_df[mapper_df['is_s7']].copy()
print(f"{len(mapper_df):,} mapper entries → {s7_mapper['file_no'].nunique():,} unique S7 IDs")
s7_mapper.head()

7,019 mapper entries → 90 unique S7 IDs


,file_no,mapper_type,docket_id,posted_date,primary_url,fallback_url,url_note,is_s7
147,S7-13-19,notice,SEC-2023-0033-0001,2023-01-17T05:00Z,https://www.sec.gov/comments/s7-13-19/s71319.htm,None,Standard reliable format.,True
247,S7-13-19,notice,SEC-2023-0036-0001,2023-01-17T05:00Z,https://www.sec.gov/comments/s7-13-19/s71319.htm,None,Standard reliable format.,True
336,S7-10-04,notice,SEC-2023-0059-0001,2023-01-23T05:00Z,https://www.sec.gov/comments/s7-10-04/s71004.htm,None,Standard reliable format.,True
337,S7-02-10,notice,SEC-2023-0059-0001,2023-01-23T05:00Z,https://www.sec.gov/comments/s7-02-10/s70210.htm,None,Standard reliable format.,True
357,S7-18-15,notice,SEC-2023-0065-0001,2023-01-24T05:00Z,https://www.sec.gov/comments/s7-18-15/s71815.htm,None,Standard reliable format.,True


In [5]:
mapper_df

,file_no,mapper_type,docket_id,posted_date,primary_url,fallback_url,url_note,is_s7
0,SR-NYSENAT-2023-04,notice,SEC-2013-0352-0003,2023-01-27T05:00Z,None,None,Exchange rule filings require scraping the SRO...,False
1,SR-NYSECHX-2022-30,notice,SEC-2022-1616-0002,2023-02-08T05:00Z,None,None,Exchange rule filings require scraping the SRO...,False
2,SR-MIAX-2022-47,notice,SEC-2023-0004-0001,2023-01-04T05:00Z,None,None,Exchange rule filings require scraping the SRO...,False
3,SR-MIAX-2018-14,notice,SEC-2023-0004-0001,2023-01-04T05:00Z,None,None,Exchange rule filings require scraping the SRO...,False
4,SR-MIAX-2019-11,notice,SEC-2023-0004-0001,2023-01-04T05:00Z,None,None,Exchange rule filings require scraping the SRO...,False
...,...,...,...,...,...,...,...,...
7029,S7-21-22,rule,SEC-2023-1398-0001,2023-12-05T05:00Z,https://www.sec.gov/comments/s7-21-22/s72122.htm,None,Standard reliable format.,True
7030,S7-01-23,rule,SEC-2023-1409-0001,2023-12-07T05:00Z,https://www.sec.gov/comments/s7-01-23/s70123.htm,None,Standard reliable format.,True
7031,S7-14-22,rule,SEC-2023-1451-0001,2023-12-15T05:00Z,https://www.sec.gov/comments/s7-14-22/s71422.htm,None,Standard reliable format.,True
7032,S7-07-23,rule,SEC-2023-1451-0001,2023-12-15T05:00Z,https://www.sec.gov/comments/s7-07-23/s70723.htm,None,Standard reliable format.,True


## Load SEC metadata CSVs

The bulk download includes CSV exports for notices, proposed rules, rules, and public submissions. We normalize the shared columns, tag the origin table, and keep the Document ID so we can join back to the mapper's `docket_id`. (The public submission CSV is empty in this drop, but the loader leaves a placeholder so downstream joins keep working.)

In [6]:
def load_metadata_tables(root: Path, names: List[str]) -> pd.DataFrame:
    frames: List[pd.DataFrame] = []
    keep_cols = [
        'Document ID',
        'Docket ID',
        'Tracking Number',
        'Title',
        'Posted Date',
        'Comment Start Date',
        'Comment Due Date',
        'Effective Date',
        'Media',
        'Abstract',
        'Content Files',
    ]
    for name in names:
        csv_path = root / f'{name}.csv'
        if not csv_path.exists() or csv_path.stat().st_size == 0:
            print(f'Skipping {name} (missing or empty)')
            continue
        frame = pd.read_csv(csv_path)
        subset_cols = [col for col in keep_cols if col in frame.columns]
        subset = frame[subset_cols].copy()
        subset = subset.rename(columns={'Document ID': 'docket_id', 'Docket ID': 'regulations_docket'})
        subset['metadata_type'] = name
        frames.append(subset)
    if not frames:
        return pd.DataFrame(columns=['docket_id'])
    return pd.concat(frames, ignore_index=True)

metadata_df = load_metadata_tables(DATA_ROOT, ['notice', 'proposed_rule', 'rule', 'public_submission'])
print(f"{len(metadata_df):,} metadata rows across {metadata_df['metadata_type'].nunique()} tables")
metadata_df.head()

Skipping public_submission (missing or empty)
1,723 metadata rows across 3 tables


,docket_id,regulations_docket,Tracking Number,Title,Posted Date,Comment Start Date,Comment Due Date,Effective Date,Media,Abstract,Content Files,metadata_type
0,SEC-2013-0352-0003,NaN,NaN,Agency Information Collection Activities; Prop...,2023-01-27T05:00Z,2023-01-27T05:00Z,NaN,NaN,NaN,NaN,https://downloads.regulations.gov/SEC-2013-035...,notice
1,SEC-2014-0104-0005,NaN,NaN,Agency Information Collection Activities; Prop...,2023-02-15T05:00Z,2023-02-15T05:00Z,NaN,NaN,NaN,NaN,https://downloads.regulations.gov/SEC-2014-010...,notice
2,SEC-2014-0104-0006,NaN,NaN,Agency Information Collection Activities; Prop...,2023-04-21T04:00Z,2023-04-21T04:00Z,NaN,NaN,NaN,NaN,https://downloads.regulations.gov/SEC-2014-010...,notice
3,SEC-2014-0589-0003,NaN,NaN,Agency Information Collection Activities; Prop...,2023-04-11T04:00Z,2023-04-11T04:00Z,NaN,NaN,NaN,NaN,https://downloads.regulations.gov/SEC-2014-058...,notice
4,SEC-2014-0589-0004,NaN,NaN,Agency Information Collection Activities; Prop...,2023-06-16T04:00Z,2023-06-16T04:00Z,NaN,NaN,NaN,NaN,https://downloads.regulations.gov/SEC-2014-058...,notice


## Load scraped S7 tables

`scraper.py` saves one JSON file per S7 docket under `scraped_tables/`. Each file has summary stats plus the parsed `usa-table` rows. We collect both the per-file summary and an exploded row-level DataFrame so downstream cells can either analyze coverage or dig into individual submissions.

In [19]:
sample_table = "../data/bulk_downloads/sec/sec_2023/scraped_tables/S7-13-19.json"

In [20]:
json.load(open(sample_table))

{'file_no': 'S7-13-19',
 'url': 'https://www.sec.gov/comments/s7-13-19/s71319.htm',
 'http_status': 200,
 'row_count': 9,
 'page_count': 1,
 'rows': [{'DateSort descending': 'Oct. 28, 2019',
   'Letter Type': 'Public Comment',
   'Commenter Name': 'Christopher Bok, Director, Financial Information Forum (FIF)',
   'comment_url': 'https://www.sec.gov/comments/s7-13-19/s71319-6355358-196251.pdf'},
  {'DateSort descending': 'Oct. 28, 2019',
   'Letter Type': 'Public Comment',
   'Commenter Name': 'Christopher Iacovella, Chief Executive Officer, American Securities Association',
   'comment_url': 'https://www.sec.gov/comments/s7-13-19/s71319-6381876-197754.pdf'},
  {'DateSort descending': 'Oct. 28, 2019',
   'Letter Type': 'Public Comment',
   'Commenter Name': 'Dennis M. Kelleher, President & CEO, and Lev Bagramian, Senior Securities Policy Advisor, Better Markets, Inc.',
   'comment_url': 'https://www.sec.gov/comments/s7-13-19/s71319-6355349-196250.pdf'},
  {'DateSort descending': 'Oct. 2

In [16]:
len(list(SCRAPED_DIR.glob('*.json')))

68

In [7]:
def load_scraped_tables(scraped_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows: List[Dict[str, object]] = []
    detail_rows: List[Dict[str, object]] = []
    for json_path in sorted(scraped_dir.glob('*.json')):
        payload = json.loads(json_path.read_text())
        rows = payload.get('rows') or []
        summary_rows.append(
            {
                'file_no': payload.get('file_no'),
                'row_count': payload.get('row_count', len(rows)),
                'page_count': payload.get('page_count'),
                'http_status': payload.get('http_status'),
                'source_url': payload.get('url'),
                'table_path': str(json_path),
            }
        )
        for idx, row in enumerate(rows):
            row_record = {'file_no': payload.get('file_no'), 'row_index': idx}
            row_record.update(row)
            detail_rows.append(row_record)
    summary_df = pd.DataFrame(summary_rows)
    detail_df = pd.DataFrame(detail_rows)
    return summary_df, detail_df

file_index_df, table_rows_df = load_scraped_tables(SCRAPED_DIR)
print(f"{len(file_index_df):,} scraped files covering {table_rows_df.shape[0]:,} total rows")
file_index_df.sort_values('row_count', ascending=False).head()

68 scraped files covering 8,718 total rows


,file_no,row_count,page_count,http_status,source_url,table_path
64,S7-32-10,1962,109.0,200.0,https://www.sec.gov/comments/s7-32-10/s73210.htm,../data/bulk_downloads/sec/sec_2023/scraped_ta...
63,S7-31-22,754,58.0,200.0,https://www.sec.gov/comments/s7-31-22/s73122.htm,../data/bulk_downloads/sec/sec_2023/scraped_ta...
3,S7-02-10,428,15.0,200.0,https://www.sec.gov/comments/s7-02-10/s70210.s...,../data/bulk_downloads/sec/sec_2023/scraped_ta...
33,S7-10-21,420,14.0,200.0,https://www.sec.gov/comments/s7-10-21/s71021.htm,../data/bulk_downloads/sec/sec_2023/scraped_ta...
61,S7-30-22,420,35.0,200.0,https://www.sec.gov/comments/s7-30-22/s73022.htm,../data/bulk_downloads/sec/sec_2023/scraped_ta...


In [18]:
table_rows_df.iloc[0]

file_no                                                         S7-01-13
row_index                                                              0
DateSort descending                                        Oct. 29, 2014
Letter Type                                   Meeting with SEC Officials
Commenter Name         Memorandum from the Office of the Investor Adv...
Comment                                                              NaN
Release Number                                                       NaN
Name: 0, dtype: object

## Join mapper, metadata, and scraper coverage

We can now align S7 IDs with their Regulations.gov Document IDs and see which dockets produced table rows. The `scrape_status` flag distinguishes dockets that produced rows, ones with empty tables, and mapper entries that never produced JSON (often due to the warnings observed when `scraper.py` could not load the HTML).

In [9]:
def compute_scrape_status(row: pd.Series) -> str:
    if pd.isna(row.get('primary_url')):
        return 'missing primary url'
    if pd.notna(row.get('table_path')) and row.get('row_count', 0) > 0:
        return 'scraped rows'
    if pd.notna(row.get('table_path')) and row.get('row_count', 0) == 0:
        return 'empty table'
    return 'not scraped'

s7_summary = (
    s7_mapper
    .merge(metadata_df, how='left', on='docket_id', suffixes=('', '_meta'))
    .merge(file_index_df, how='left', on='file_no')
    .sort_values('file_no')
    .reset_index(drop=True)
)
s7_summary['row_count'] = s7_summary['row_count'].fillna(0).astype(int)
s7_summary['page_count'] = s7_summary['page_count'].fillna(0).astype(int)
s7_summary['scrape_status'] = s7_summary.apply(compute_scrape_status, axis=1)

coverage = s7_summary['scrape_status'].value_counts().to_frame('count')
coverage

,count
scrape_status,
scraped rows,191
empty table,57
not scraped,21
missing primary url,1


In [10]:
# Inspect a few representative dockets with their joined metadata
columns_to_show = [
    'file_no',
    'scrape_status',
    'metadata_type',
    'Title',
    'posted_date',
    'Comment Start Date',
    'Comment Due Date',
    'row_count',
    'page_count',
    'primary_url',
]
s7_summary[columns_to_show].head(12)

,file_no,scrape_status,metadata_type,Title,posted_date,Comment Start Date,Comment Due Date,row_count,page_count,primary_url
0,S7-01-13,scraped rows,notice,Self-Regulatory Organizations; Proposed Rule C...,2023-04-13T04:00Z,2023-04-13T04:00Z,NaN,85,3,https://www.sec.gov/comments/s7-01-13/s70113.htm
1,S7-01-13,scraped rows,proposed_rule,Disclosure of Order Execution Information,2023-01-20T05:00Z,2023-01-20T05:00Z,2023-04-01T03:59:59Z,85,3,https://www.sec.gov/comments/s7-01-13/s70113.htm
2,S7-01-13,scraped rows,notice,Self-Regulatory Organizations; Proposed Rule C...,2023-02-21T05:00Z,2023-02-21T05:00Z,NaN,85,3,https://www.sec.gov/comments/s7-01-13/s70113.htm
3,S7-01-22,scraped rows,rule,Form PF; Event Reporting for Large Hedge Fund ...,2023-06-12T04:00Z,2023-06-12T04:00Z,NaN,145,5,https://www.sec.gov/comments/s7-01-22/s70122.htm
4,S7-01-22,scraped rows,rule,Money Market Fund Reforms; Form PF Reporting R...,2023-08-03T04:00Z,2023-08-03T04:00Z,NaN,145,5,https://www.sec.gov/comments/s7-01-22/s70122.htm
5,S7-01-23,scraped rows,rule,Prohibition Against Conflicts of Interest in C...,2023-12-07T05:00Z,2023-12-07T05:00Z,NaN,108,27,https://www.sec.gov/comments/s7-01-23/s70123.htm
6,S7-01-23,scraped rows,proposed_rule,Prohibition against Conflicts of Interest in C...,2023-02-14T05:00Z,2023-02-14T05:00Z,2023-03-28T03:59:59Z,108,27,https://www.sec.gov/comments/s7-01-23/s70123.htm
7,S7-02-10,scraped rows,proposed_rule,Disclosure of Order Execution Information,2023-01-20T05:00Z,2023-01-20T05:00Z,2023-04-01T03:59:59Z,428,15,https://www.sec.gov/comments/s7-02-10/s70210.htm
8,S7-02-10,scraped rows,notice,Self-Regulatory Organizations; Proposed Rule C...,2023-03-24T04:00Z,2023-03-24T04:00Z,NaN,428,15,https://www.sec.gov/comments/s7-02-10/s70210.htm
9,S7-02-10,scraped rows,notice,Self-Regulatory Organizations; Proposed Rule C...,2023-04-19T04:00Z,2023-04-19T04:00Z,NaN,428,15,https://www.sec.gov/comments/s7-02-10/s70210.htm


### Which S7 IDs still need attention?

The table below surfaces mapper entries where we either never produced a JSON table or the `mapper` entry is missing a primary URL. This is a quick way to revisit the `[WARN] Failed to fetch ...` messages from the CLI run.

In [11]:
needs_attention = s7_summary[s7_summary['scrape_status'].isin(['not scraped', 'missing primary url'])]
needs_attention[['file_no', 'docket_id', 'metadata_type', 'Title', 'primary_url']].head(15)

,file_no,docket_id,metadata_type,Title,primary_url
31,S7-03-13,SEC-2023-0824-0001,rule,Money Market Fund Reforms; Form PF Reporting R...,https://www.sec.gov/comments/s7-03-13/s70313.htm
60,S7-03-22,SEC-2023-0996-0001,rule,Private Fund Advisers: Documentation of Regist...,https://www.sec.gov/comments/s7-03-22/s70322.htm
83,S7-050-22,SEC-2023-0404-0001,notice,Self-Regulatory Organizations; Proposed Rule C...,https://www.sec.gov/comments/s7-050-22/s705022...
98,S7-08-09,SEC-2023-1244-0001,rule,Short Position and Short Activity Reporting by...,https://www.sec.gov/comments/s7-08-09/s70809.htm
106,S7-09-22,SEC-2023-0835-0001,rule,"Cybersecurity Risk Management, Strategy, Gover...",https://www.sec.gov/comments/s7-09-22/s70922.htm
164,S7-11-03,SEC-2023-0990-0001,notice,Joint Industry Plan: National Market System Pl...,https://www.sec.gov/comments/s7-11-03/s71103.htm
165,S7-11-2022,SEC-2023-0647-0001,rule,Removal of References to Credit Ratings from R...,https://www.sec.gov/comments/s7-11-2022/s71120...
167,S7-11-22,SEC-2023-0647-0001,rule,Removal of References to Credit Ratings from R...,https://www.sec.gov/comments/s7-11-22/s71122.htm
173,S7-12-21,SEC-2023-1044-0001,proposed_rule,"Electronic Data Gathering, Analysis, and Retri...",https://www.sec.gov/comments/s7-12-21/s71221.htm
174,S7-12-23,SEC-2023-0850-0001,proposed_rule,Conflicts of Interest Associated with the Use ...,https://www.sec.gov/comments/s7-12-23/s71223.htm


## Row-level data joins

`table_rows_df` keeps one record per row scraped from the SEC HTML tables. Joining back to `s7_summary` provides titles and comment windows so you can pivot or filter without re-reading the JSON files.

In [12]:
if table_rows_df.empty:
    print('No row-level data found in scraped_tables/')
else:
    enriched_rows = table_rows_df.merge(
        s7_summary[['file_no', 'Title', 'Comment Start Date', 'Comment Due Date', 'metadata_type']],
        on='file_no',
        how='left',
    )
    display(enriched_rows.head())

    example_file = 'S7-31-08'
    example_rows = enriched_rows[enriched_rows['file_no'] == example_file]
    if not example_rows.empty:
        print(f'Example rows for {example_file}:')
        display(example_rows.head(10))
    else:
        print(f'No rows available for {example_file}')

,file_no,row_index,DateSort descending,Letter Type,Commenter Name,Comment,Release Number,Title,Comment Start Date,Comment Due Date,metadata_type
0,S7-01-13,0,"Oct. 29, 2014",Meeting with SEC Officials,Memorandum from the Office of the Investor Adv...,NaN,NaN,Self-Regulatory Organizations; Proposed Rule C...,2023-04-13T04:00Z,NaN,notice
1,S7-01-13,0,"Oct. 29, 2014",Meeting with SEC Officials,Memorandum from the Office of the Investor Adv...,NaN,NaN,Disclosure of Order Execution Information,2023-01-20T05:00Z,2023-04-01T03:59:59Z,proposed_rule
2,S7-01-13,0,"Oct. 29, 2014",Meeting with SEC Officials,Memorandum from the Office of the Investor Adv...,NaN,NaN,Self-Regulatory Organizations; Proposed Rule C...,2023-02-21T05:00Z,NaN,notice
3,S7-01-13,1,"Oct. 29, 2014",Meeting with SEC Officials,Memorandum from the Office of the Investor Adv...,NaN,NaN,Self-Regulatory Organizations; Proposed Rule C...,2023-04-13T04:00Z,NaN,notice
4,S7-01-13,1,"Oct. 29, 2014",Meeting with SEC Officials,Memorandum from the Office of the Investor Adv...,NaN,NaN,Disclosure of Order Execution Information,2023-01-20T05:00Z,2023-04-01T03:59:59Z,proposed_rule


Example rows for S7-31-08:


,file_no,row_index,DateSort descending,Letter Type,Commenter Name,Comment,Release Number,Title,Comment Start Date,Comment Due Date,metadata_type
22149,S7-31-08,0,"Dec. 30, 2009",Public Comment,"F. William Capp, President and CEO, Beacon Pow...",NaN,NaN,Short Position and Short Activity Reporting by...,2023-11-01T04:00Z,NaN,rule
22150,S7-31-08,1,"Dec. 17, 2009",Public Comment,John Wages,NaN,NaN,Short Position and Short Activity Reporting by...,2023-11-01T04:00Z,NaN,rule
22151,S7-31-08,2,"June 11, 2009",Public Comment,Dave Patch,NaN,NaN,Short Position and Short Activity Reporting by...,2023-11-01T04:00Z,NaN,rule
22152,S7-31-08,3,"May 9, 2009",Public Comment,Mark Faulk,NaN,NaN,Short Position and Short Activity Reporting by...,2023-11-01T04:00Z,NaN,rule
22153,S7-31-08,4,"March 25, 2009",Public Comment,"Roel C. Campos, Esq., Partner in Charge, Coole...",NaN,NaN,Short Position and Short Activity Reporting by...,2023-11-01T04:00Z,NaN,rule
22154,S7-31-08,5,"March 15, 2009",Public Comment,Reza Ganjavi,NaN,NaN,Short Position and Short Activity Reporting by...,2023-11-01T04:00Z,NaN,rule
22155,S7-31-08,6,"Jan. 23, 2009",Public Comment,Alain Engelen,NaN,NaN,Short Position and Short Activity Reporting by...,2023-11-01T04:00Z,NaN,rule
22156,S7-31-08,7,"Jan. 13, 2009",Public Comment,"F. William Capp, President and CEO, Beacon Pow...",NaN,NaN,Short Position and Short Activity Reporting by...,2023-11-01T04:00Z,NaN,rule
22157,S7-31-08,8,"Jan. 11, 2009",Public Comment,"Reza Ganjavi, Musician, Scientist, Philosopher...",NaN,NaN,Short Position and Short Activity Reporting by...,2023-11-01T04:00Z,NaN,rule
22158,S7-31-08,9,"Jan. 9, 2009",Public Comment,"Chris Harris, Ph.D., Private Investor, Lexingt...",NaN,NaN,Short Position and Short Activity Reporting by...,2023-11-01T04:00Z,NaN,rule


## Next steps

* Filter `enriched_rows` to focus on specific comment attributes (e.g., `Commenter Name`) and feed them into downstream NLP pipelines.
* Export `s7_summary` to CSV so harvesting gaps (missing URLs or failed fetches) can be queued for retries.
* If you need additional SEC context, join `regulations_docket` back to the rest of the Regulations.gov data lake or embed the downloaded PDFs/HTML from `downloaded_content/`.
